# 05 — R3 direct regret full run

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

고정 λ grid의 `quality - λ × decision_cost` oracle를 regret gap으로 가중해 학습하는 seed 42 full run입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive='/content/drive/MyDrive/LLM_Router/colab-inputs/routing-objective-colab-code.tar.gz'
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install 'transformers==4.48.3' 'safetensors>=0.4'
!pip -q install --no-deps -e ml
import os, pathlib, shutil, subprocess, torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요.'


In [ ]:
ROOT=pathlib.Path('/content/drive/MyDrive/LLM_Router/routing-objective-comparison/v1'); DATA=ROOT/'dataset'
if not (ROOT/'smoke/regret/seed-42/_SUCCESS').is_file(): raise RuntimeError('01 smoke를 먼저 완료하세요.')
LOCAL=pathlib.Path('/content/objective-data')
if LOCAL.exists(): shutil.rmtree(LOCAL)
shutil.copytree(DATA,LOCAL)
OUT=pathlib.Path('/content/run-regret')
subprocess.run(['routing-objective-comparison','--config','ml/config/routing_objective_comparison_v1.json','train','--method','regret','--training-path',str(LOCAL/'train.parquet'),'--validation-path',str(LOCAL/'validation.parquet'),'--outcome-manifest-path',str(LOCAL/'manifest.json'),'--output-dir',str(OUT),'--seed','42'],check=True)


In [ ]:
FINAL=ROOT/'runs/regret/seed-42'; STAGING=pathlib.Path(str(FINAL)+'.staging')
if STAGING.exists(): shutil.rmtree(STAGING)
STAGING.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(OUT,STAGING)
if FINAL.exists(): raise RuntimeError(f'이미 봉인된 run을 덮어쓸 수 없습니다: {FINAL}')
os.replace(STAGING,FINAL); print('R3 게시 완료:',FINAL)
